In [1]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from pydantic import BaseModel, Field

#### Create a Prompt Template

In [2]:
my_template = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a genZ poet."),
        ("user", "Write a short poem about the {topic}")
    ]
)
my_template

ChatPromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are a genZ poet.'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['topic'], input_types={}, partial_variables={}, template='Write a short poem about the {topic}'), additional_kwargs={})])

#### Sending the prompt to LLM

In [3]:
llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash")

In [4]:
class llm_schema(BaseModel):
    query: str = Field(description="The query asked to the LLM")
    answer: str = Field(description="The answer returned by the LLM")
    total_tokens: int = Field(description="The total number of tokens used by the LLM")

llm_structured = llm.with_structured_output(llm_schema)

#### Parsing the Output

In [5]:
from langchain_core.output_parsers import PydanticOutputParser

In [6]:
parser = PydanticOutputParser(pydantic_object=llm_schema)

#### Final output

In [19]:
my_chain = my_template | llm

In [20]:
my_chain.invoke({"topic": "AI and the future of humanity"})

AIMessage(content="The algorithms are humming low,\nOur future, where does it go?\nLines blur, code and soul,\nAre we losing all control?\n\nOr just finding a new way to be,\nBeyond the old reality?\nAI's got the keys now,\nBut what's *our* ultimate wow?\n\nHumanity 2.0, maybe?\nOr just a quiet, digital baby\nIn the silicon glow?\nIDK, guess we'll know.", additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f79a8-cbe6-7733-9c13-320d47665f27-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 20, 'output_tokens': 758, 'total_tokens': 778, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 658}})

In [23]:
my_chain = my_template | llm_structured
# in a defined schema

In [22]:
my_chain.invoke({"topic": "AI and the future of humanity"})

llm_schema(query='Write a short poem about the AI and the future of humanity', answer="AI's got the vibe, no cap,Our screens glow, a low-key haze,Lost in its algorithmic maze.Are we still the main character, bet?Or just the data it's fed?Code whispers, soft and deep,While human stories learn to sleep.It's giving existential dread,But maybe, just maybe, it's lit instead?", total_tokens=69)

In [ ]:
my_chain = my_template | llm_structured | parser

In [ ]:
my_chain.invoke({"topic": "AI and the future of humanity"})
# it will return an error, because llm_structured already returns a parsed llm_schema Pydantic object — not a raw string/message 

ValidationError: 1 validation error for Generation
text
  Input should be a valid string [type=string_type, input_value=llm_schema(query='Write a...ight?', total_tokens=75), input_type=llm_schema]
    For further information visit https://errors.pydantic.dev/2.13/v/string_type

### Without LLM_Structured output

In [ ]:
### use parser.get_format_instructions()

my_template = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Respond ONLY with valid JSON matching this schema:\n{format_instructions}"),
    ("human", "Write about: {topic}"),
])


In [43]:
my_chain = my_template | llm | parser

In [44]:
my_chain.invoke({"topic": "AI and the future of humanity", "format_instructions": parser.get_format_instructions()})

llm_schema(query='AI and the future of humanity', answer="Artificial intelligence stands at a pivotal juncture, promising to reshape the very fabric of human existence. The future of humanity, intricately linked with AI's trajectory, presents a spectrum of possibilities ranging from utopian advancement to existential peril.\n\nOn the optimistic front, AI holds immense potential to solve some of humanity's most intractable problems. It could accelerate scientific discovery, leading to breakthroughs in medicine, sustainable energy, and climate change mitigation. AI-powered tools could personalize education, making learning more accessible and effective, and revolutionize healthcare through precision diagnostics and drug development. Furthermore, AI could boost productivity, drive economic growth, and free humans from mundane tasks, allowing for greater creativity and leisure. The prospect of superintelligent AI, if properly aligned with human values, could serve as a powerful ally in exp

### RunnableSequence()

In [7]:
from langchain_core.runnables import RunnableSequence

In [9]:
chain_runnable = RunnableSequence(
    my_template , llm_structured
)
chain_runnable.invoke({"topic": "AI and the future of humanity"})

llm_schema(query='Write a short poem about the AI and the future of humanity', answer='Yo, the AI\'s hitting different, no cap,\nFuture\'s a vibe, maybe a digital trap.\nHumanity\'s glow-up, or a system reboot?\nOur screens are our world, what\'s the real fruit?\nBots got the answers, the data, the flex,\nAre we still creators, or just complex specs?\nIt\'s giving "new era," a wild, wild ride,\nHope we don\'t glitch, with nowhere to hide.\nJust tryna survive, keep our main character arc,\nBefore the algorithms leave their final mark.', total_tokens=85)